# V2 Inventory Intelligence — Historical Backtesting & Simulation

## Objective

Version 1 answers:

> **"What should I order today?"**

Version 2 asks a more important business question:

> **"If I had followed this inventory policy in the past, would it actually have worked?"**

To answer this, V2 simulates inventory decisions over a historical period.

For each day in the backtest:

1. Use only information that would have been available on that day.
2. Generate a demand forecast.
3. Apply the existing V1 inventory policy.
4. Place an order when the reorder condition is triggered.
5. Wait for the supplier lead time.
6. Receive arriving purchase orders.
7. Fulfill the actual historical demand.
8. Record the resulting inventory state.

The same historical demand and inventory policy are then tested using:

- **XGBoost forecast**
- **7-day Moving Average baseline**

The two strategies are compared using:

- Inventory levels
- Stockouts and lost sales
- Service level
- Ordering activity
- Holding cost
- Ordering cost
- Stockout cost
- Total inventory cost

### Business question

The final goal is not simply to find the most accurate forecast.

It is to determine:

> **Which forecasting strategy leads to the better inventory decision for the business?**

## Experiment Design

### Product

The initial V2 experiment uses:

- **Product:** P001 — Wireless Headphones

### Backtest Period

**2025-10-01 to 2025-12-31**

This is the historical test period used to evaluate the inventory system.

### Strategies

#### Strategy 1 — XGBoost

Uses the trained V1 XGBoost forecasting model and the existing V1 inventory policy.

#### Strategy 2 — Moving Average

Uses a simple 7-day moving average forecast with the **same inventory policy**.

### Important Principle

Both strategies use:

- The same historical demand
- The same starting inventory
- The same lead time
- The same safety stock
- The same reorder logic
- The same target inventory policy
- The same order-arrival mechanics

This allows us to isolate the impact of the forecasting method.

## 1. Setup & Data

In [1]:
 # ============================================================
# V2 Inventory Simulation
# ============================================================

import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.inventory.policy import (
    calculate_target_inventory,
    calculate_recommended_order_qty,
)

from src.inventory.reorder import (
    calculate_reorder_point,
    should_reorder,
)

from src.inventory.safety_stock import (
    calculate_safety_stock,
)

from src.inventory.simulation import (
    run_backtest,
    get_arrivals_for_date,
    process_daily_demand,
    create_purchase_order,
    calculate_inventory_position,
)

In [3]:
sales = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "sales.csv"
)

sales["date"] = pd.to_datetime(sales["date"])


print("Sales shape:", sales.shape)
print("Sales date range:", sales["date"].min(), "to", sales["date"].max())
print("Products:", sales["product_id"].unique())

Sales shape: (3655, 11)
Sales date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Products: <StringArray>
['P001', 'P002', 'P003', 'P004', 'P005']
Length: 5, dtype: str


In [4]:
product_id = "P001"

product_history = (
    sales[sales["product_id"] == product_id]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

print("Product:", product_id)
print("Product history shape:", product_history.shape)
print(
    "Product history:",
    product_history["date"].min(),
    "to",
    product_history["date"].max(),
)

Product: P001
Product history shape: (731, 11)
Product history: 2024-01-01 00:00:00 to 2025-12-31 00:00:00


## 2. Forecasting Model & Backtest Configuration

In [5]:
MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "xgboost_forecaster.joblib"
)

model = joblib.load(MODEL_PATH)

MODEL_FEATURES = [
    "price",
    "discount",
    "promotion",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
]

print("Model loaded:", type(model))

Model loaded: <class 'xgboost.sklearn.XGBRegressor'>


In [6]:
backtest_start = pd.Timestamp("2025-10-01")
backtest_end = pd.Timestamp("2025-12-31")

lead_time_days = 4
inventory_days = 5

print("Backtest:", backtest_start.date(), "to", backtest_end.date())
print("Lead time:", lead_time_days, "days")
print("Inventory days:", inventory_days)

Backtest: 2025-10-01 to 2025-12-31
Lead time: 4 days
Inventory days: 5


## 3. Starting Inventory & Safety Stock

The simulation needs two important starting values:

- **Starting inventory:** how many units we assume are available on 2025-10-01.
- **Safety stock:** extra inventory maintained to protect against forecast error during supplier lead time.

Starting inventory is estimated using the average historical daily demand before the backtest and the V1 inventory-days setting.

Safety stock is taken from the V1 forecast-error calculation for the selected product.

In [7]:
historical_before_backtest = product_history[
    product_history["date"] < backtest_start
].copy()

average_daily_demand = (
    historical_before_backtest["units_sold"].mean()
)

starting_stock = int(
    round(average_daily_demand * inventory_days)
)

print("Average daily demand:", round(average_daily_demand, 2))
print("Starting stock:", starting_stock)

Average daily demand: 45.42
Starting stock: 227


In [8]:
forecast_error_std = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "forecast_error_std.csv"
)

p001_error_std = float(
    forecast_error_std.loc[
        forecast_error_std["product_id"] == product_id,
        "error_std"
    ].iloc[0]
)

print("P001 forecast error std:", round(p001_error_std, 4))

P001 forecast error std: 5.5941


In [9]:
safety_stock = calculate_safety_stock(
    error_std=p001_error_std,
    lead_time_days=lead_time_days,
)

print("Safety stock:", safety_stock)

Safety stock: 19.0


In [10]:
print("Product:", product_id)
print("Lead time:", lead_time_days)
print("Safety stock:", safety_stock)
print("Starting stock:", starting_stock)
print("Backtest:", backtest_start.date(), "to", backtest_end.date())

Product: P001
Lead time: 4
Safety stock: 19.0
Starting stock: 227
Backtest: 2025-10-01 to 2025-12-31


## 4. XGBoost Inventory Backtest

The first V2 strategy uses the trained V1 XGBoost forecasting model.

For every day in the historical backtest period, the simulator:

1. Uses only information available before that day.
2. Generates a 30-day demand forecast.
3. Calculates lead-time demand.
4. Applies the V1 reorder policy.
5. Places an order when required.
6. Receives orders after the configured lead time.
7. Fulfills the actual historical demand.
8. Records the inventory outcome.

This allows us to evaluate the complete forecasting + inventory decision system rather than forecasting accuracy alone.

In [11]:
backtest_results = run_backtest(
    product_history=product_history,
    start_date=backtest_start,
    end_date=backtest_end,
    starting_stock=starting_stock,
    model=model,
    model_features=MODEL_FEATURES,
    safety_stock=safety_stock,
    lead_time_days=lead_time_days,
)

print("XGBoost backtest shape:", backtest_results.shape)

XGBoost backtest shape: (92, 15)


In [12]:
backtest_results.head(10)

,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,total_forecast,lead_time_demand,safety_stock,reorder_point,inventory_position,reorder_required,target_inventory,order_qty
0,2025-10-01,227,0,39,39,0,188,831.936707,134.351395,19.0,153.351395,188,False,850.936707,0
1,2025-10-02,188,0,27,27,0,161,835.219482,140.279938,19.0,159.279938,161,False,854.219482,0
2,2025-10-03,161,0,45,45,0,116,821.565796,138.809296,19.0,157.809296,116,True,840.565796,725
3,2025-10-04,116,0,40,40,0,76,833.714111,137.855713,19.0,156.855713,801,False,852.714111,0
4,2025-10-05,76,0,35,35,0,41,820.384216,131.314224,19.0,150.314224,766,False,839.384216,0
5,2025-10-06,41,0,36,36,0,5,793.311890,123.703949,19.0,142.703949,730,False,812.311890,0
6,2025-10-07,5,725,29,29,0,701,793.535767,123.470863,19.0,142.470863,701,False,812.535767,0
7,2025-10-08,701,0,40,40,0,661,788.257690,125.822227,19.0,144.822227,661,False,807.257690,0
8,2025-10-09,661,0,46,46,0,615,802.409119,130.099762,19.0,149.099762,615,False,821.409119,0
9,2025-10-10,615,0,53,53,0,562,819.333008,130.072739,19.0,149.072739,562,False,838.333008,0


## 5. Moving Average Baseline

To understand whether XGBoost actually improves the inventory system, we need a simple benchmark.

The baseline uses a **7-day moving average** of historical demand.

For each backtest day:

- Only the previous 7 days of actual demand are used.
- The daily forecast is multiplied by 30 to create the 30-day forecast.
- The same V1 reorder point, safety stock, target inventory, lead time, and order logic are applied.

The only difference between the two strategies is the forecasting method.

In [13]:
baseline_results = []

current_stock_baseline = starting_stock
purchase_orders_baseline = []

for current_date in backtest_results["date"]:

    # Use only information available before today
    history_before_today = product_history[
        product_history["date"] < current_date
    ].copy()

    # 7-day moving average forecast
    baseline_daily_forecast = (
        history_before_today["units_sold"]
        .tail(7)
        .mean()
    )

    baseline_total_forecast = (
        baseline_daily_forecast * 30
    )

    baseline_lead_time_demand = (
        baseline_daily_forecast * lead_time_days
    )

    # Receive purchase orders arriving today
    arrival_qty = get_arrivals_for_date(
        purchase_orders=purchase_orders_baseline,
        date=current_date,
    )

    available_stock = (
        current_stock_baseline + arrival_qty
    )

    # Actual historical demand
    actual_demand = int(
        product_history.loc[
            product_history["date"] == current_date,
            "units_sold"
        ].iloc[0]
    )

    # Consume today's demand
    (
        units_fulfilled,
        stockout_units,
        closing_stock,
    ) = process_daily_demand(
        available_stock=available_stock,
        demand=actual_demand,
    )

    # Inventory position after today's demand
    inventory_position = calculate_inventory_position(
        current_stock=closing_stock,
        purchase_orders=purchase_orders_baseline,
        current_date=current_date,
    )

    # Reorder decision
    reorder_point = calculate_reorder_point(
        lead_time_demand=baseline_lead_time_demand,
        safety_stock=safety_stock,
    )

    reorder_required = should_reorder(
        inventory_position=inventory_position,
        reorder_point=reorder_point,
    )

    # Target inventory
    target_inventory = calculate_target_inventory(
        total_forecast=baseline_total_forecast,
        safety_stock=safety_stock,
    )

    # Recommended order quantity
    order_qty = calculate_recommended_order_qty(
        target_inventory=target_inventory,
        inventory_position=inventory_position,
        reorder_required=reorder_required,
    )

    order_qty = float(
        np.asarray(order_qty).item()
    )

    # Create order if required
    if order_qty > 0:
        new_order = create_purchase_order(
            order_date=current_date,
            quantity=int(order_qty),
            lead_time_days=lead_time_days,
        )

        purchase_orders_baseline.append(new_order)

    # Save daily result
    baseline_results.append(
        {
            "date": current_date,
            "opening_stock": current_stock_baseline,
            "arrival_qty": arrival_qty,
            "demand": actual_demand,
            "units_fulfilled": units_fulfilled,
            "stockout_units": stockout_units,
            "closing_stock": closing_stock,
            "baseline_daily_forecast": baseline_daily_forecast,
            "baseline_total_forecast": baseline_total_forecast,
            "baseline_lead_time_demand": baseline_lead_time_demand,
            "safety_stock": safety_stock,
            "reorder_point": reorder_point,
            "inventory_position": inventory_position,
            "reorder_required": reorder_required,
            "target_inventory": target_inventory,
            "order_qty": int(order_qty),
        }
    )

    current_stock_baseline = closing_stock

baseline_results = pd.DataFrame(baseline_results)

print("Baseline backtest shape:", baseline_results.shape)

Baseline backtest shape: (92, 16)


In [14]:
# Moving Average baseline performance metrics

baseline_avg_inventory = (
    baseline_results["closing_stock"].mean()
)

baseline_max_inventory = (
    baseline_results["closing_stock"].max()
)

baseline_total_units_ordered = (
    baseline_results["order_qty"].sum()
)

baseline_number_of_orders = (
    (baseline_results["order_qty"] > 0).sum()
)

baseline_stockout_days = (
    (baseline_results["stockout_units"] > 0).sum()
)

baseline_total_lost_sales = (
    baseline_results["stockout_units"].sum()
)

baseline_total_demand = (
    baseline_results["demand"].sum()
)

baseline_total_fulfilled = (
    baseline_results["units_fulfilled"].sum()
)

baseline_service_level = (
    baseline_total_fulfilled
    / baseline_total_demand
    * 100
)

print("Moving Average Baseline Performance")
print("------------------------------------")
print("Average inventory:", round(baseline_avg_inventory, 2))
print("Maximum inventory:", baseline_max_inventory)
print("Total units ordered:", baseline_total_units_ordered)
print("Number of orders:", baseline_number_of_orders)
print("Stockout days:", baseline_stockout_days)
print("Lost sales units:", baseline_total_lost_sales)
print("Total demand:", baseline_total_demand)
print("Units fulfilled:", baseline_total_fulfilled)
print("Service level:", round(baseline_service_level, 2), "%")

Moving Average Baseline Performance
------------------------------------
Average inventory: 544.41
Maximum inventory: 1261
Total units ordered: 4139
Number of orders: 4
Stockout days: 0
Lost sales units: 0
Total demand: 3879
Units fulfilled: 3879
Service level: 100.0 %


In [15]:
# V2 cost assumptions

holding_cost_rate = 0.20
ordering_cost_per_order = 500
stockout_cost_per_unit = 1000
unit_cost = 1000

backtest_days_count = len(backtest_results)

print("Holding cost rate:", holding_cost_rate)
print("Ordering cost per order: ₹", ordering_cost_per_order)
print("Stockout cost per unit: ₹", stockout_cost_per_unit)
print("Unit cost: ₹", unit_cost)
print("Backtest days:", backtest_days_count)

Holding cost rate: 0.2
Ordering cost per order: ₹ 500
Stockout cost per unit: ₹ 1000
Unit cost: ₹ 1000
Backtest days: 92


In [16]:
# XGBoost inventory cost

xgb_avg_inventory = (
    backtest_results["closing_stock"].mean()
)

xgb_number_of_orders = (
    (backtest_results["order_qty"] > 0).sum()
)

xgb_total_lost_units = (
    backtest_results["stockout_units"].sum()
)

xgb_average_inventory_value = (
    xgb_avg_inventory * unit_cost
)

xgb_holding_cost = (
    xgb_average_inventory_value
    * holding_cost_rate
    * backtest_days_count
    / 365
)

xgb_ordering_cost = (
    xgb_number_of_orders
    * ordering_cost_per_order
)

xgb_stockout_cost = (
    xgb_total_lost_units
    * stockout_cost_per_unit
)

xgb_total_inventory_cost = (
    xgb_holding_cost
    + xgb_ordering_cost
    + xgb_stockout_cost
)

print("XGBoost Inventory Cost")
print("----------------------")
print("Average inventory:", round(xgb_avg_inventory, 2))
print("Holding cost: ₹", round(xgb_holding_cost, 2))
print("Ordering cost: ₹", round(xgb_ordering_cost, 2))
print("Stockout cost: ₹", round(xgb_stockout_cost, 2))
print(
    "Total inventory cost: ₹",
    round(xgb_total_inventory_cost, 2)
)

XGBoost Inventory Cost
----------------------
Average inventory: 374.92
Holding cost: ₹ 18900.27
Ordering cost: ₹ 2500
Stockout cost: ₹ 26000
Total inventory cost: ₹ 47400.27


In [17]:
# Moving Average inventory cost

baseline_avg_inventory = (
    baseline_results["closing_stock"].mean()
)

baseline_number_of_orders = (
    (baseline_results["order_qty"] > 0).sum()
)

baseline_total_lost_units = (
    baseline_results["stockout_units"].sum()
)

baseline_average_inventory_value = (
    baseline_avg_inventory * unit_cost
)

baseline_holding_cost = (
    baseline_average_inventory_value
    * holding_cost_rate
    * backtest_days_count
    / 365
)

baseline_ordering_cost = (
    baseline_number_of_orders
    * ordering_cost_per_order
)

baseline_stockout_cost = (
    baseline_total_lost_units
    * stockout_cost_per_unit
)

baseline_total_inventory_cost = (
    baseline_holding_cost
    + baseline_ordering_cost
    + baseline_stockout_cost
)

print("Moving Average Inventory Cost")
print("-----------------------------")
print("Average inventory:", round(baseline_avg_inventory, 2))
print("Holding cost: ₹", round(baseline_holding_cost, 2))
print("Ordering cost: ₹", round(baseline_ordering_cost, 2))
print("Stockout cost: ₹", round(baseline_stockout_cost, 2))
print(
    "Total inventory cost: ₹",
    round(baseline_total_inventory_cost, 2)
)

Moving Average Inventory Cost
-----------------------------
Average inventory: 544.41
Holding cost: ₹ 27444.38
Ordering cost: ₹ 2000
Stockout cost: ₹ 0
Total inventory cost: ₹ 29444.38


In [18]:
# Final V2 strategy comparison

comparison = pd.DataFrame(
    {
        "Metric": [
            "Average Inventory",
            "Maximum Inventory",
            "Total Units Ordered",
            "Number of Orders",
            "Stockout Days",
            "Lost Sales Units",
            "Service Level (%)",
            "Holding Cost (₹)",
            "Ordering Cost (₹)",
            "Stockout Cost (₹)",
            "Total Inventory Cost (₹)",
        ],
        "XGBoost": [
            xgb_avg_inventory,
            backtest_results["closing_stock"].max(),
            backtest_results["order_qty"].sum(),
            xgb_number_of_orders,
            (backtest_results["stockout_units"] > 0).sum(),
            xgb_total_lost_units,
            (
                backtest_results["units_fulfilled"].sum()
                / backtest_results["demand"].sum()
                * 100
            ),
            xgb_holding_cost,
            xgb_ordering_cost,
            xgb_stockout_cost,
            xgb_total_inventory_cost,
        ],
        "Moving Average": [
            baseline_avg_inventory,
            baseline_results["closing_stock"].max(),
            baseline_results["order_qty"].sum(),
            baseline_number_of_orders,
            (baseline_results["stockout_units"] > 0).sum(),
            baseline_total_lost_units,
            (
                baseline_results["units_fulfilled"].sum()
                / baseline_results["demand"].sum()
                * 100
            ),
            baseline_holding_cost,
            baseline_ordering_cost,
            baseline_stockout_cost,
            baseline_total_inventory_cost,
        ],
    }
)

comparison

,Metric,XGBoost,Moving Average
0,Average Inventory,374.923913,544.413043
1,Maximum Inventory,822.000000,1261.000000
2,Total Units Ordered,3949.000000,4139.000000
3,Number of Orders,5.000000,4.000000
4,Stockout Days,2.000000,0.000000
5,Lost Sales Units,26.000000,0.000000
6,Service Level (%),99.329724,100.000000
7,Holding Cost (₹),18900.273973,27444.383562
8,Ordering Cost (₹),2500.000000,2000.000000
9,Stockout Cost (₹),26000.000000,0.000000


In [19]:
# Final V2 business conclusion

xgb_cost = xgb_total_inventory_cost
baseline_cost = baseline_total_inventory_cost

cost_difference = abs(xgb_cost - baseline_cost)
cheaper_strategy = (
    "XGBoost"
    if xgb_cost < baseline_cost
    else "Moving Average"
)

print("V2 BUSINESS CONCLUSION")
print("======================")
print()
print(f"Backtest period: {backtest_start.date()} to {backtest_end.date()}")
print(f"Product: {product_id}")
print()
print(f"XGBoost total inventory cost: ₹{xgb_cost:,.2f}")
print(f"Moving Average total inventory cost: ₹{baseline_cost:,.2f}")
print()
print(
    f"Cheaper strategy: {cheaper_strategy}"
)
print(
    f"Cost difference: ₹{cost_difference:,.2f}"
)
print()
print(
    f"XGBoost average inventory: "
    f"{xgb_avg_inventory:.2f} units"
)
print(
    f"Moving Average average inventory: "
    f"{baseline_avg_inventory:.2f} units"
)
print()
print(
    f"XGBoost service level: "
    f"{(
        backtest_results['units_fulfilled'].sum()
        / backtest_results['demand'].sum()
        * 100
    ):.2f}%"
)
print(
    f"Moving Average service level: "
    f"{(
        baseline_results['units_fulfilled'].sum()
        / baseline_results['demand'].sum()
        * 100
    ):.2f}%"
)

V2 BUSINESS CONCLUSION

Backtest period: 2025-10-01 to 2025-12-31
Product: P001

XGBoost total inventory cost: ₹47,400.27
Moving Average total inventory cost: ₹29,444.38

Cheaper strategy: Moving Average
Cost difference: ₹17,955.89

XGBoost average inventory: 374.92 units
Moving Average average inventory: 544.41 units

XGBoost service level: 99.33%
Moving Average service level: 100.00%
